# Retrieving additional metadata on movies using the API of TMDB

This script uses the API of The Movie Database in retrieving additional information on movies predefined in previous scripts. Additional information is collected on ratings and vote counts, ... . The API documentation can be consulted at; https://developer.themoviedb.org/reference/movie-details

In [1]:
import pandas as pd

#Read in the films from the existing dataset created in R
moviedata = pd.read_csv("../data/raw/raw_tmdb.csv")

print(moviedata.columns)

tmdb_id = moviedata['id'].unique().tolist()
print(f"\nAmount of movies in the dataset: {len(tmdb_id)}")

C:\Users\stefv\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Index(['tconst', 'id', 'title_sanitycheck', 'TMDB_rating', 'TMDB_votecount',
       'timestamp1', 'budget', 'production_companies', 'production_countries',
       'timestamp2'],
      dtype='str')

Amount of movies in the dataset: 1009


In [2]:
import requests
import json
import time
import os

# API key
api_key = os.environ['TMDBkey']

extradata = []

for movie in tmdb_id:
    # API endpoint
    url = f"https://api.themoviedb.org/3/movie/{movie}/release_dates"
    
    # Parameters
    params = {
        "api_key": api_key
    }
    
    # Make the request
    response = requests.get(url, params=params)
    
    # Check if successful
    if response.status_code == 200:
        data = response.json()  # data IS the movie object directly

        # Confirm movie
        new_id = movie
        print(f"MOVIE FOUND with ID: {movie}")

        # Initialize release dates
        cinema_release = None
        digital_release = None
        
        # Retrieve release dates
        results = data.get("results", [])
        
        # Loop through countries to find US
        for country in results:
            if country["iso_3166_1"] == "US":
                # Loop through release dates for US
                release_dates = country.get("release_dates", [])
                
                for release in release_dates:
                    # Type 3 = Theatrical
                    if release["type"] == 3:
                        cinema_release = release["release_date"]
                    # Type 4 = Digital
                    elif release["type"] == 4:
                        digital_release = release["release_date"]
                
                break  # Stop after finding US
        
        # Confirm new data
        extradata.append({
            "id": new_id,
            "cinema_release": cinema_release,
            "digital_release": digital_release,
            "timestamp3": time.time()
        })
    else:
        print(f"Error {response.status_code}: Movie {movie} not found")
    
    time.sleep(.027)


# Convert new data to DataFrame
df_new = pd.DataFrame(extradata)

# Merge the two DataFrames on the id/tconst column
df_merged = pd.merge(moviedata, df_new, on='id', how='left')

# Save back to CSV (overwrite the original)
df_merged.to_csv('../data/raw/raw_tmdb.csv', index=False)

print(f"\nAdded new columns at {time.ctime(time.time())}!")

MOVIE FOUND with ID: 116745
MOVIE FOUND with ID: 169917
MOVIE FOUND with ID: 45772
MOVIE FOUND with ID: 76489
MOVIE FOUND with ID: 504562
MOVIE FOUND with ID: 49529
MOVIE FOUND with ID: 49849
MOVIE FOUND with ID: 38778
MOVIE FOUND with ID: 34544
MOVIE FOUND with ID: 77883
MOVIE FOUND with ID: 39254
MOVIE FOUND with ID: 10193
MOVIE FOUND with ID: 399579
MOVIE FOUND with ID: 72976
MOVIE FOUND with ID: 283350
MOVIE FOUND with ID: 287947
MOVIE FOUND with ID: 417859
MOVIE FOUND with ID: 87827
MOVIE FOUND with ID: 29427
MOVIE FOUND with ID: 1771
MOVIE FOUND with ID: 43593
MOVIE FOUND with ID: 44754
MOVIE FOUND with ID: 59108
MOVIE FOUND with ID: 41513
MOVIE FOUND with ID: 9543
MOVIE FOUND with ID: 64685
MOVIE FOUND with ID: 34813
MOVIE FOUND with ID: 48988
MOVIE FOUND with ID: 72431
MOVIE FOUND with ID: 49022
MOVIE FOUND with ID: 35169
MOVIE FOUND with ID: 417644
MOVIE FOUND with ID: 38843
MOVIE FOUND with ID: 43347
MOVIE FOUND with ID: 72358
MOVIE FOUND with ID: 63492
MOVIE FOUND with ID: 4